# Dependency-Tracked Cache Invalidation for Multilingual RAG

Companion notebook to `cache_invalidation.py` in this repo. Demonstrates
why chunk-level, dependency-tracked invalidation beats whole-document
invalidation and TTL-based expiry.

Full writeup: "Your RAG System Is Right Today. Tomorrow It's Confidently Wrong."
(Article 5 in the LinkedIn series — link added once published.)

This notebook is self-contained: it uses lightweight stand-in
chunker/embedder/vector-store classes so it runs without any external
API keys. Swap them for the real `chunker`, `embedder`, and
`vector_store` objects from `multilingual_rag_pipeline.ipynb` to use
this against the actual Hindi demo corpus and embedding model.

In [1]:
from cache_invalidation import (
    DocumentRecord, AnswerCache, diff_chunks, on_document_updated, compute_hash
)
from datetime import datetime

## 1. Minimal stand-in components

These mimic the interfaces the real pipeline components expose
(`chunker.split`, `embedder.embed`, `vector_store.upsert/delete`),
so `on_document_updated` can be demoed without any live embedding calls.

In [2]:
class DemoChunker:
    """Splits on blank lines. The real pipeline uses the recursive/semantic
    chunker from the main notebook; this is a stand-in for demo purposes."""
    def split(self, content: str) -> dict:
        parts = content.split("\n\n")
        return {f"chunk_{i}": p for i, p in enumerate(parts) if p.strip()}


class DemoEmbedder:
    """Placeholder embedder. Swap for the BGE/Cohere multilingual embedder
    used elsewhere in this repo."""
    def embed(self, text: str):
        return [len(text)]


class DemoVectorStore:
    def __init__(self):
        self.store = {}
    def upsert(self, cid, vec):
        self.store[cid] = vec
    def delete(self, cid):
        self.store.pop(cid, None)

## 2. Synthetic Hindi policy document (v1)

Reuses the same style of synthetic Hindi demo content as the main
pipeline notebook — a short refund/shipping policy split into two
paragraphs (two chunks).

In [3]:
hindi_policy_v1 = (
    "\u0930\u093f\u092b\u0902\u0921 \u0928\u0940\u0924\u093f: "
    "\u0917\u094d\u0930\u093e\u0939\u0915 \u0916\u0930\u0940\u0926 "
    "\u0915\u0940 \u0924\u093e\u0930\u0940\u0916 \u0938\u0947 30 "
    "\u0926\u093f\u0928\u094b\u0902 \u0915\u0947 \u0905\u0902\u0926\u0930 "
    "\u0930\u093f\u092b\u0902\u0921 \u0915\u0947 \u0932\u093f\u090f "
    "\u092a\u093e\u0924\u094d\u0930 \u0939\u0948\u0902\u0964"
    "\n\n"
    "\u0936\u093f\u092a\u093f\u0902\u0917 \u0928\u0940\u0924\u093f: "
    "\u0921\u093f\u0932\u0940\u0935\u0930\u0940 \u092e\u0947\u0902 "
    "5-7 \u0915\u093e\u0930\u094d\u092f \u0926\u093f\u0935\u0938 "
    "\u0932\u0917\u0924\u0947 \u0939\u0948\u0902\u0964"
)

print(hindi_policy_v1)

रिफंड नीति: ग्राहक खरीद की तारीख से 30 दिनों के अंदर रिफंड के लिए पात्र हैं।

शिपिंग नीति: डिलीवरी में 5-7 कार्य दिवस लगते हैं।


## 3. Initial ingest

Ingest the document, then simulate caching two answers generated
from it, each tagged with the chunk id it actually depended on.

In [4]:
doc_store = {}
answer_cache = AnswerCache()
chunker, embedder, vector_store = DemoChunker(), DemoEmbedder(), DemoVectorStore()

result = on_document_updated(
    doc_id="hindi_refund_policy",
    new_content=hindi_policy_v1,
    doc_store=doc_store,
    chunker=chunker,
    embedder=embedder,
    vector_store=vector_store,
    answer_cache=answer_cache,
)
print("Initial ingest:", result)

answer_cache.store("q_refund_window_hi", "30 din", ["chunk_0"])
answer_cache.store("q_shipping_time_hi", "5-7 din", ["chunk_1"])

print("Cached queries:", list(answer_cache.cache.keys()))

Initial ingest: {'doc_id': 'hindi_refund_policy', 'chunks_re_embedded': 2, 'chunks_removed': 0, 'answers_purged': 0}
Cached queries: ['q_refund_window_hi', 'q_shipping_time_hi']


## 4. Document is edited — refund window changes from 30 to 45 days

Only the shipping clause is untouched. A correct system should
re-embed only the refund chunk and purge only the refund answer.

In [5]:
hindi_policy_v2 = hindi_policy_v1.replace("30", "45")

updated_result = on_document_updated(
    doc_id="hindi_refund_policy",
    new_content=hindi_policy_v2,
    doc_store=doc_store,
    chunker=chunker,
    embedder=embedder,
    vector_store=vector_store,
    answer_cache=answer_cache,
)
print("After edit:", updated_result)

print("Refund answer still cached? ", answer_cache.get("q_refund_window_hi") is not None)
print("Shipping answer still cached?", answer_cache.get("q_shipping_time_hi") is not None)

After edit: {'doc_id': 'hindi_refund_policy', 'chunks_re_embedded': 1, 'chunks_removed': 0, 'answers_purged': 1}
Refund answer still cached?  False
Shipping answer still cached? True


**Expected result:** 1 chunk re-embedded, 0 removed, 1 answer purged —
the refund answer is gone, the shipping answer survives untouched.

This is the core claim from the article: invalidation cost and
staleness risk should scale with *how much actually changed*, not
with document size or a fixed TTL clock.

## 5. Contrast: what whole-document invalidation would have cost

For comparison, here's what a naive whole-document invalidation
strategy would purge for the same edit — every chunk and every
cached answer tied to the document, regardless of relevance.

In [6]:
naive_chunks_reembedded = len(doc_store["hindi_refund_policy"].chunk_ids)
naive_answers_purged = 2  # both cached answers, since whole-doc invalidation is blunt

print(f"Chunk-level invalidation : {updated_result['chunks_re_embedded']} chunk re-embedded, "
      f"{updated_result['answers_purged']} answer purged")
print(f"Whole-document invalidation (naive): {naive_chunks_reembedded} chunks re-embedded, "
      f"{naive_answers_purged} answers purged")

Chunk-level invalidation : 1 chunk re-embedded, 1 answer purged
Whole-document invalidation (naive): 2 chunks re-embedded, 2 answers purged
